# Cognitive Decline Detection — Exploration

Three things in here:
- feature distributions (AD vs HC per group)
- ablation results (which feature group matters)
- model comparison + feature importance

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

from src.dataset import load_dataset
from src.features import GROUPS

AUDIO_DIR = "../data/adress/audio"
LABELS    = "../data/adress/labels.csv"
RESULTS   = "../results/scores.csv"

In [ ]:
X_groups, y, ids = load_dataset(AUDIO_DIR, LABELS)
print(f"N={len(y)}  AD={y.sum()}  HC={(y==0).sum()}")

## Feature distributions

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
titles = {"mfcc": "MFCC", "pitch": "Pitch (F0)", "energy": "Energy + Speech Rate"}

for ax, g in zip(axes, ["mfcc", "pitch", "energy"]):
    X = X_groups[g]
    ax.boxplot([X[y==0].mean(axis=1), X[y==1].mean(axis=1)], labels=["HC", "AD"])
    ax.set_title(titles[g])

plt.suptitle("Feature group means: AD vs HC", y=1.02)
plt.tight_layout()
plt.savefig("../results/feature_distributions.png", dpi=150, bbox_inches="tight")
plt.show()

## Ablation results

In [ ]:
df = pd.read_csv(RESULTS)
abl = df[df["exp"] == "ablation"][["variant", "acc", "acc_std", "f1", "auc"]].set_index("variant")
abl

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(abl.index, abl["acc"], yerr=abl["acc_std"], capsize=4,
       color=["#4C72B0", "#DD8452", "#55A868", "#C44E52"])
ax.axhline(0.5, linestyle="--", color="gray", linewidth=0.8)
ax.set_ylim(0.4, 1.0)
ax.set_ylabel("Accuracy (5-fold CV)")
ax.set_title("Feature ablation — SVM")
plt.tight_layout()
plt.savefig("../results/ablation.png", dpi=150, bbox_inches="tight")
plt.show()

## Model comparison

In [ ]:
df[df["exp"] == "models"][["model", "acc", "acc_std", "f1", "auc"]].set_index("model")

## Feature importance (RF)

In [ ]:
pipe = Pipeline([("sc", StandardScaler()), ("rf", RandomForestClassifier(n_estimators=200, random_state=42))])
pipe.fit(X_groups["all"], y)
imp = pipe.named_steps["rf"].feature_importances_
top = np.argsort(imp)[::-1][:15]

fig, ax = plt.subplots(figsize=(9, 3))
ax.bar(range(15), imp[top])
ax.set_xticks(range(15))
ax.set_xticklabels([f"f{i}" for i in top], rotation=45)
ax.set_title("Top 15 feature importances (RF)")
plt.tight_layout()
plt.savefig("../results/feature_importance.png", dpi=150, bbox_inches="tight")
plt.show()

## Confusion matrix (SVM, all features)

In [ ]:
pipe_svm = Pipeline([("sc", StandardScaler()), ("clf", SVC(kernel="rbf", random_state=42))])
y_pred = cross_val_predict(pipe_svm, X_groups["all"], y,
                            cv=StratifiedKFold(5, shuffle=True, random_state=42))

fig, ax = plt.subplots(figsize=(4, 4))
ConfusionMatrixDisplay(confusion_matrix(y, y_pred), display_labels=["HC", "AD"]).plot(ax=ax, colorbar=False)
ax.set_title("SVM — 5-fold CV")
plt.tight_layout()
plt.savefig("../results/confusion_matrix.png", dpi=150, bbox_inches="tight")
plt.show()